```markdown
### Téléchargement du Dataset OTTO depuis Kaggle

Pour télécharger ce dataset, vous devez d'abord obtenir un jeton API de Kaggle (`kaggle.json`) :
1. Allez sur votre profil Kaggle -> **Settings** -> **Create New Token**.
2. Importez le fichier `kaggle.json` dans votre espace Colab ou utilisez le formulaire ci-dessous.
```

```markdown
### Téléchargement alternatif avec KaggleHub

`kagglehub` permet de télécharger facilement des jeux de données ou des compétitions Kaggle directement via Python.
```

In [ ]:
# Installation ou mise à jour de kagglehub
!pip install -q kagglehub

In [7]:
import kagglehub

# Téléchargement des fichiers de la compétition 'otto-recommender-system'
path = kagglehub.competition_download('otto-recommender-system')

print("Le dataset a été téléchargé avec succès à l'emplacement :", path)

100%|██████████| 1.94G/1.94G [01:34<00:00, 21.9MB/s]

Extracting files...


Le dataset a été téléchargé avec succès à l'emplacement : /root/.cache/kagglehub/competitions/otto-recommender-system


In [8]:
import os
# Lister les fichiers téléchargés pour vérification
if os.path.exists(path):
    print("Fichiers disponibles :", os.listdir(path))

Fichiers disponibles : ['sample_submission.csv', 'train.jsonl', 'test.jsonl']


### Exploration des fichiers Train et Test

Les données sont stockées au format JSON Lines (`.jsonl`). Chaque ligne correspond à un document JSON représentant l'historique d'une session utilisateur.

In [23]:
import json
import pandas as pd

train_path = os.path.join(path, 'train.jsonl')
test_path = os.path.join(path, 'test.jsonl')

# Lecture des premières lignes avec pandas de façon efficace
# 'lines=True' indique que chaque ligne est un objet JSON distinct
# 'chunksize' permet de ne lire qu'un bloc spécifique de données

# Lecture d'un échantillon du fichier d'entraînement (train)
with pd.read_json(train_path, lines=True, chunksize=5) as reader:
    train_preview = next(reader)

# Lecture d'un échantillon du fichier de test
with pd.read_json(test_path, lines=True, chunksize=5) as reader:
    test_preview = next(reader)

print("Aperçu du DataFrame d'entraînement (Train) :")
display(train_preview)

print("\nAperçu du DataFrame de test (Test) :")
display(test_preview)

Aperçu du DataFrame d'entraînement (Train) :


,session,events
0,0,"[{'aid': 1517085, 'ts': 1659304800025, 'type':..."
1,1,"[{'aid': 424964, 'ts': 1659304800025, 'type': ..."
2,2,"[{'aid': 763743, 'ts': 1659304800038, 'type': ..."
3,3,"[{'aid': 1425967, 'ts': 1659304800095, 'type':..."
4,4,"[{'aid': 613619, 'ts': 1659304800119, 'type': ..."



Aperçu du DataFrame de test (Test) :


,session,events
0,12899779,"[{'aid': 59625, 'ts': 1661724000278, 'type': '..."
1,12899780,"[{'aid': 1142000, 'ts': 1661724000378, 'type':..."
2,12899781,"[{'aid': 141736, 'ts': 1661724000559, 'type': ..."
3,12899782,"[{'aid': 1669402, 'ts': 1661724000568, 'type':..."
4,12899783,"[{'aid': 255297, 'ts': 1661724000572, 'type': ..."


## Feature Engineering : 1. Caractéristiques au niveau de la Session

Nous allons concevoir des indicateurs clés pour chaque session d'utilisateur (durée, volume d'activité, types d'actions).

In [11]:
def extract_session_features(df):
    """Extrait des caractéristiques statistiques au niveau de chaque session."""
    session_features = []

    for idx, row in df.iterrows():
        session_id = row['session']
        events = row['events']

        # Nombre total d'événements
        total_events = len(events)

        # Extraction des types et des timestamps
        types = [e['type'] for e in events]
        timestamps = [e['ts'] for e in events]

        # Durée de la session en secondes
        duration_sec = (max(timestamps) - min(timestamps)) / 1000.0 if total_events > 1 else 0.0

        # Comptage par type d'événement
        num_clicks = types.count('clicks')
        num_carts = types.count('carts')
        num_orders = types.count('orders')

        session_features.append({
            'session': session_id,
            'total_events': total_events,
            'session_duration_sec': duration_sec,
            'num_clicks': num_clicks,
            'num_carts': num_carts,
            'num_orders': num_orders
        })

    return pd.DataFrame(session_features)

# Application sur notre aperçu pour validation
session_features_df = extract_session_features(train_preview)
display(session_features_df)

,session,total_events,session_duration_sec,num_clicks,num_carts,num_orders
0,0,276,2380183.682,255,17,4
1,1,32,2410054.967,24,8,0
2,2,33,2409415.621,32,1,0
3,3,226,1804866.676,200,21,5
4,4,19,2281881.184,15,3,1


## Feature Engineering : 2. Caractéristiques des Produits (AIDs)

In [12]:
def extract_product_features(df):
    """Calcule les statistiques de popularité pour chaque produit (AID)."""
    # On "aplatit" les événements pour avoir une ligne par interaction produit
    all_events = []
    for idx, row in df.iterrows():
        for event in row['events']:
            all_events.append({
                'aid': event['aid'],
                'type': event['type']
            })

    events_df = pd.DataFrame(all_events)

    # Groupement par produit (aid) pour compter l'occurrence de chaque type d'action
    product_stats = events_df.groupby('aid')['type'].value_counts().unstack(fill_value=0)

    # S'assurer que toutes les colonnes d'actions existent
    for action in ['clicks', 'carts', 'orders']:
        if action not in product_stats.columns:
            product_stats[action] = 0

    product_stats['total_interactions'] = product_stats['clicks'] + product_stats['carts'] + product_stats['orders']
    return product_stats.reset_index()

product_features_df = extract_product_features(train_preview)
display(product_features_df.head())

type,aid,carts,clicks,orders,total_interactions
0,2512,0,1,0,1
1,3999,0,1,0,1
2,6851,0,1,0,1
3,7563,0,2,0,2
4,10268,0,1,0,1


## Feature Engineering : 3. Séquences d'interactions

Nous allons extraire le dernier produit visité dans la session ainsi que l'avant-dernier afin de capturer l'effet de récence temporelle.

In [13]:
def extract_sequence_features(df):
    """Extrait les derniers éléments visités dans chaque session."""
    sequence_features = []
    for idx, row in df.iterrows():
        session_id = row['session']
        events = row['events']

        # Liste ordonnée des produits visités (AID)
        aid_sequence = [e['aid'] for e in events]

        last_aid = aid_sequence[-1] if len(aid_sequence) >= 1 else None
        second_last_aid = aid_sequence[-2] if len(aid_sequence) >= 2 else None
        unique_aids_count = len(set(aid_sequence))

        sequence_features.append({
            'session': session_id,
            'last_aid': last_aid,
            'second_last_aid': second_last_aid,
            'unique_aids_count': unique_aids_count
        })

    return pd.DataFrame(sequence_features)

sequence_features_df = extract_sequence_features(train_preview)
display(sequence_features_df)

,session,last_aid,second_last_aid,unique_aids_count
0,0,161938,1740927,183
1,1,497868,376932,22
2,2,672473,808782,29
3,3,925352,925352,140
4,4,479396,1081407,12


## Préparation des données pour LightGBM (Reranking / Classification Binaire)

Nous allons construire un dataset d'entraînement en associant chaque couple `(session, aid)` à ses caractéristiques (features) et à une étiquette cible indiquant si le produit a été acheté (`orders`) ou mis au panier (`carts`).

In [14]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# 1. Préparation de la table de base (session, aid)
rows = []
for idx, row in train_preview.iterrows():
    session_id = row['session']
    events = row['events']
    # Pour chaque interaction dans la session
    for event in events:
        rows.append({
            'session': session_id,
            'aid': event['aid'],
            'target': 1 if event['type'] in ['carts', 'orders'] else 0
        })

df_train = pd.DataFrame(rows).drop_duplicates(subset=['session', 'aid'])

# 2. Fusionner avec nos features créées précédemment
# Caractéristiques de Session
df_train = df_train.merge(session_features_df, on='session', how='left')

# Caractéristiques de Produit
df_train = df_train.merge(product_features_df, on='aid', how='left')

# Caractéristiques de Séquence
df_train = df_train.merge(sequence_features_df, on='session', how='left')

# Création d'une feature dynamique : Est-ce que cet AID est le dernier vu dans la séquence ?
df_train['is_last_aid'] = (df_train['aid'] == df_train['last_aid']).astype(int)

# Remplissage des valeurs manquantes
df_train.fillna(0, inplace=True)

display(df_train.head())

,session,aid,target,total_events,session_duration_sec,num_clicks,num_carts,num_orders,carts,clicks,orders,total_interactions,last_aid,second_last_aid,unique_aids_count,is_last_aid
0,0,1517085,0,276,2380183.682,255,17,4,0,1,0,1,161938,1740927,183,0
1,0,1563459,0,276,2380183.682,255,17,4,0,1,0,1,161938,1740927,183,0
2,0,1309446,0,276,2380183.682,255,17,4,0,1,0,1,161938,1740927,183,0
3,0,16246,0,276,2380183.682,255,17,4,0,1,0,1,161938,1740927,183,0
4,0,1781822,0,276,2380183.682,255,17,4,0,1,0,1,161938,1740927,183,0


In [25]:
# Étape 1 : Séparation chronologique des sessions brutes
# On divise chaque session en une partie passée (historique) et future (cibles)

def split_session_chronologically(df, split_ratio=0.8):
    history_records = []
    future_records = []

    for idx, row in df.iterrows():
        session_id = row['session']
        events = row['events']
        n_events = len(events)

        if n_events < 2:
            # Pas assez d'événements pour diviser, on garde tout dans l'historique
            history_records.append({'session': session_id, 'events': events})
            continue

        # Point de coupure chronologique
        split_point = int(n_events * split_ratio)
        if split_point == 0:
            split_point = 1

        history_events = events[:split_point]
        future_events = events[split_point:]

        history_records.append({'session': session_id, 'events': history_events})
        future_records.append({'session': session_id, 'events': future_events})

    return pd.DataFrame(history_records), pd.DataFrame(future_records)

# Séparation de notre échantillon d'entraînement
train_history, train_future = split_session_chronologically(train_preview)

print(f"Séparation effectuée avec succès !")
print(f"  - Nombre de sessions dans l'historique : {len(train_history)}")
print(f"  - Nombre de sessions dans le futur : {len(train_future)}")

Séparation effectuée avec succès !
  - Nombre de sessions dans l'historique : 5
  - Nombre de sessions dans le futur : 5


## Étape 2 : Feature Engineering STRICTEMENT sur le passé (`train_history`)

Nous calculons nos indicateurs de session, de produit, et de séquences de manière isolée pour éviter toute fuite de données future.

In [27]:
# Extraction des variables explicatives uniquement sur l'historique de l'échantillon
history_session_feat = extract_session_features(train_history)
history_product_feat = extract_product_features(train_history)
history_sequence_feat = extract_sequence_features(train_history)

print("Extraction des caractéristiques sur l'historique terminée avec succès !")
print(f"  - Caractéristiques de sessions : {history_session_feat.shape}")
print(f"  - Caractéristiques de produits : {history_product_feat.shape}")
print(f"  - Caractéristiques de séquences : {history_sequence_feat.shape}")

Extraction des caractéristiques sur l'historique terminée avec succès !
  - Caractéristiques de sessions : (5, 6)
  - Caractéristiques de produits : (304, 5)
  - Caractéristiques de séquences : (5, 4)


## Étape 3 : Construction des Candidats et Association des Cibles Futures

Pour chaque session, nous listons les produits visités dans le passé (nos candidats). Ensuite, nous regardons s'ils ont fait l'objet d'une action (`clicks`, `carts`, `orders`) dans le futur de la session (`train_future`) pour poser nos étiquettes cibles.

In [29]:
# 1. Identification de tous les couples (session, aid) candidats issus de l'historique
rows_clean = []
for idx, row in train_history.iterrows():
    session_id = row['session']
    events = row['events']
    for event in events:
        rows_clean.append({
            'session': session_id,
            'aid': event['aid']
        })

df_train_clean = pd.DataFrame(rows_clean).drop_duplicates(subset=['session', 'aid'])

# 2. Détermination des étiquettes cibles à partir du futur
future_targets = []
for idx, row in train_future.iterrows():
    session_id = row['session']
    events = row['events']
    for event in events:
        future_targets.append({
            'session': session_id,
            'aid': event['aid'],
            'future_click': 1 if event['type'] == 'clicks' else 0,
            'future_cart': 1 if event['type'] == 'carts' else 0,
            'future_order': 1 if event['type'] == 'orders' else 0
        })

if len(future_targets) > 0:
    df_future_targets = pd.DataFrame(future_targets).groupby(['session', 'aid']).max().reset_index()
    df_train_clean = df_train_clean.merge(df_future_targets, on=['session', 'aid'], how='left')
else:
    df_train_clean['future_click'] = 0
    df_train_clean['future_cart'] = 0
    df_train_clean['future_order'] = 0

df_train_clean.fillna(0, inplace=True)

# 3. Fusion étanche avec les caractéristiques calculées sur l'historique
df_train_clean = df_train_clean.merge(history_session_feat, on='session', how='left')
df_train_clean = df_train_clean.merge(history_product_feat, on='aid', how='left')
df_train_clean = df_train_clean.merge(history_sequence_feat, on='session', how='left')
df_train_clean['is_last_aid'] = (df_train_clean['aid'] == df_train_clean['last_aid']).astype(int)
df_train_clean.fillna(0, inplace=True)

print("Dataset d'entraînement propre généré !")
display(df_train_clean.head())

Dataset d'entraînement propre généré !


,session,aid,future_click,future_cart,future_order,total_events,session_duration_sec,num_clicks,num_carts,num_orders,carts,clicks,orders,total_interactions,last_aid,second_last_aid,unique_aids_count,is_last_aid
0,0,1517085,0.0,0.0,0.0,220,2247375.149,205,13,2,0,1,0,1,442293,543308,148,0
1,0,1563459,0.0,0.0,0.0,220,2247375.149,205,13,2,0,1,0,1,442293,543308,148,0
2,0,1309446,0.0,0.0,0.0,220,2247375.149,205,13,2,0,1,0,1,442293,543308,148,0
3,0,16246,0.0,0.0,0.0,220,2247375.149,205,13,2,0,1,0,1,442293,543308,148,0
4,0,1781822,0.0,0.0,0.0,220,2247375.149,205,13,2,0,1,0,1,442293,543308,148,0


## Étape 4 : Entraînement et Évaluation Réaliste (Sans Fuite de Données)

Nous entraînons nos 3 modèles sur les cibles futures et mesurons de manière réaliste la performance Recall@20.

In [32]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import numpy as np

# Définition des variables explicatives (features) et cibles saines
features = [
    'total_events', 'session_duration_sec', 'num_clicks', 'num_carts', 'num_orders',
    'clicks', 'carts', 'orders', 'total_interactions', 'unique_aids_count', 'is_last_aid'
]

targets_clean = {
    'clicks': 'future_click',
    'carts': 'future_cart',
    'orders': 'future_order'
}

models_clean = {}

# Séparation train/validation basée sur les indices propres de df_train_clean
train_idx_clean, val_idx_clean = train_test_split(df_train_clean.index, test_size=0.2, random_state=42)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 15,
    'verbose': -1
}

for obj, target_col in targets_clean.items():
    print(f"--- Entraînement : {obj} (sans leakage) ---")
    X_tr = df_train_clean.loc[train_idx_clean, features]
    y_tr = df_train_clean.loc[train_idx_clean, target_col]
    X_va = df_train_clean.loc[val_idx_clean, features]
    y_va = df_train_clean.loc[val_idx_clean, target_col]

    tr_data = lgb.Dataset(X_tr, label=y_tr)
    va_data = lgb.Dataset(X_va, label=y_va, reference=tr_data)

    models_clean[obj] = lgb.train(params, tr_data, num_boost_round=50, valid_sets=[va_data])

# Fonction d'évaluation propre du Recall@20
def evaluate_recall_at_20_clean(df_val_set, models_dict, features_list):
    df_eval = df_val_set.copy()
    results = {}
    for obj, model in models_dict.items():
        df_eval[f'pred_{obj}'] = model.predict(df_eval[features_list])
        recalls = []
        for session_id, group in df_eval.groupby('session'):
            target_col = f'future_click' if obj == 'clicks' else f'future_cart' if obj == 'carts' else f'future_order'
            true_positives = set(group[group[target_col] == 1]['aid'])
            if len(true_positives) == 0:
                continue
            top_20_predicted = set(group.sort_values(by=f'pred_{obj}', ascending=False).head(20)['aid'])
            hits = len(true_positives.intersection(top_20_predicted))
            recalls.append(hits / len(true_positives))
        results[obj] = np.mean(recalls) if recalls else 0.0
    return results

df_validation_clean = df_train_clean.loc[val_idx_clean]
scores_recall_clean = evaluate_recall_at_20_clean(df_validation_clean, models_clean, features)

print("\n--- Résultats du Recall@20 Corrigés (Sans Leakage) ---")
for obj, score in scores_recall_clean.items():
    print(f"Recall@20 pour {obj:<8} : {score:.4f}")

--- Entraînement : clicks (sans leakage) ---
--- Entraînement : carts (sans leakage) ---
--- Entraînement : orders (sans leakage) ---

--- Résultats du Recall@20 Corrigés (Sans Leakage) ---
Recall@20 pour clicks   : 1.0000
Recall@20 pour carts    : 1.0000
Recall@20 pour orders   : 0.0000


## Solution : Entraînement et évaluation sur un échantillon plus grand (50 000 sessions)

Pour obtenir des scores de Recall@20 réalistes, nous appliquons exactement la même logique sur un volume de données suffisant.

In [35]:
import json
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# 1. Chargement de 50 000 sessions
print("Chargement de 50 000 sessions...")
with pd.read_json(train_path, lines=True, chunksize=50000) as reader:
    train_large = next(reader)

# 2. Séparation chronologique (past/future)
print("Séparation chronologique...")
train_history_lg, train_future_lg = split_session_chronologically(train_large)

# 3. Extraction des features sur le passé uniquement
print("Calcul des variables explicatives (Features)...")
history_session_feat_lg = extract_session_features(train_history_lg)
history_product_feat_lg = extract_product_features(train_history_lg)
history_sequence_feat_lg = extract_sequence_features(train_history_lg)

# 4. Construction des candidats (session, aid) et association des cibles
print("Génération du dataset d'entraînement...")
rows_lg = []
for idx, row in train_history_lg.iterrows():
    session_id = row['session']
    for event in row['events']:
        rows_lg.append({'session': session_id, 'aid': event['aid']})

df_train_lg = pd.DataFrame(rows_lg).drop_duplicates(subset=['session', 'aid'])

future_targets_lg = []
for idx, row in train_future_lg.iterrows():
    session_id = row['session']
    for event in row['events']:
        future_targets_lg.append({
            'session': session_id,
            'aid': event['aid'],
            'future_click': 1 if event['type'] == 'clicks' else 0,
            'future_cart': 1 if event['type'] == 'carts' else 0,
            'future_order': 1 if event['type'] == 'orders' else 0
        })

if len(future_targets_lg) > 0:
    df_future_targets_lg = pd.DataFrame(future_targets_lg).groupby(['session', 'aid']).max().reset_index()
    df_train_lg = df_train_lg.merge(df_future_targets_lg, on=['session', 'aid'], how='left')
else:
    df_train_lg['future_click'] = 0
    df_train_lg['future_cart'] = 0
    df_train_lg['future_order'] = 0

df_train_lg.fillna(0, inplace=True)

# Fusion avec les features
df_train_lg = df_train_lg.merge(history_session_feat_lg, on='session', how='left')
df_train_lg = df_train_lg.merge(history_product_feat_lg, on='aid', how='left')
df_train_lg = df_train_lg.merge(history_sequence_feat_lg, on='session', how='left')
df_train_lg['is_last_aid'] = (df_train_lg['aid'] == df_train_lg['last_aid']).astype(int)
df_train_lg.fillna(0, inplace=True)

# 5. Entraînement et Évaluation
print("Entraînement des modèles...")
train_idx_lg, val_idx_lg = train_test_split(df_train_lg.index, test_size=0.2, random_state=42)

models_lg = {}
for obj, target_col in targets_clean.items():
    X_tr = df_train_lg.loc[train_idx_lg, features]
    y_tr = df_train_lg.loc[train_idx_lg, target_col]
    X_va = df_train_lg.loc[val_idx_lg, features]
    y_va = df_train_lg.loc[val_idx_lg, target_col]

    tr_data = lgb.Dataset(X_tr, label=y_tr)
    va_data = lgb.Dataset(X_va, label=y_va, reference=tr_data)

    models_lg[obj] = lgb.train(params, tr_data, num_boost_round=50, valid_sets=[va_data])

print("Calcul des scores Recall@20 réels...")
df_validation_lg = df_train_lg.loc[val_idx_lg]
scores_recall_lg = evaluate_recall_at_20_clean(df_validation_lg, models_lg, features)

print("\n--- Nouveaux Résultats du Recall@20 (Échelle Réelle) ---")
for obj, score in scores_recall_lg.items():
    print(f"Recall@20 pour {obj:<8} : {score:.4f}")

Chargement de 50 000 sessions...
Séparation chronologique...
Calcul des variables explicatives (Features)...
Génération du dataset d'entraînement...
Entraînement des modèles...
Calcul des scores Recall@20 réels...

--- Nouveaux Résultats du Recall@20 (Échelle Réelle) ---
Recall@20 pour clicks   : 0.9775
Recall@20 pour carts    : 0.9776
Recall@20 pour orders   : 0.9911
